# 📈 Chapter 12: Least Squares Applications and Extensions

---

## 12.1 Introduction: Pushing the Limits of Least Squares

In the previous chapter, we learned that the Ordinary Least Squares (OLS) algorithm finds the optimal weights ($\beta$) for a linear model by projecting the target data onto the column space of the design matrix $\mathbf{X}$. 

However, real-world data is rarely perfectly linear, and real-world features are rarely perfectly independent. In this chapter, we will push the limits of the General Linear Model by addressing two major applications:
1. **Polynomial Regression:** How can we use *Linear* Algebra to fit *Non-Linear* curves?
2. **Multicollinearity & Regularization:** What happens when our features are too similar to each other, causing the matrix to become unstable, and how do we fix it using Ridge Regression (Tikhonov Regularization)?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Set plotting style
%matplotlib inline
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 12.")

## 12.2 Polynomial Regression via Linear Algebra

It sounds like an oxymoron: using *Linear* models to fit *Curves*. But in linear algebra, "linear" refers to the relationship between the **weights ($\beta$)**, not the features themselves. 

If we have a single feature $x$ (e.g., time) but the target $y$ follows a curved trajectory, we can simply engineer new features by squaring or cubing the original feature. Our design matrix $\mathbf{X}$ will look like this:

$$ \mathbf{X} = \begin{bmatrix} 1 & x_1 & x_1^2 & x_1^3 \\ 1 & x_2 & x_2^2 & x_2^3 \\ \vdots & \vdots & \vdots & \vdots \end{bmatrix} $$

Even though the columns contain non-linear transformations of $x$, the matrix equation $\mathbf{X}\beta = \mathbf{y}$ remains perfectly linear with respect to $\beta$. We can solve it using the exact same Least Squares method!

In [ ]:
# 1. Generate Non-Linear Data (A cubic curve with noise)
n_samples = 100
x = np.linspace(-3, 3, n_samples)
y_true = 1.5 - 2.0 * x + 0.5 * x**2 + 1.2 * x**3
y_noisy = y_true + np.random.normal(0, 3, n_samples)

# 2. Construct the Polynomial Design Matrix (Degree 3)
intercept = np.ones(n_samples)
x_linear = x
x_squared = x**2
x_cubed = x**3

X_poly = np.column_stack((intercept, x_linear, x_squared, x_cubed))
print(f"Polynomial Design Matrix Shape: {X_poly.shape}")

# 3. Solve using Least Squares (QR Decomposition backend via NumPy)
beta_poly, _, _, _ = np.linalg.lstsq(X_poly, y_noisy, rcond=None)
print(f"\nCalculated Betas: {np.round(beta_poly, 2)}")
print(f"Original Betas  : [1.5, -2.0, 0.5, 1.2]")

# 4. Calculate predictions
y_pred = X_poly @ beta_poly

# 5. Visualization
plt.figure(figsize=(8, 5))
plt.scatter(x, y_noisy, color='gray', alpha=0.6, label='Noisy Data')
plt.plot(x, y_pred, color='red', linewidth=3, label='Polynomial Fit (Degree 3)')
plt.title('Fitting Non-Linear Data using Linear Algebra')
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 12.3 Multicollinearity and the Condition Number

What happens when two columns in our design matrix are almost identical? For example, predicting house prices using 'Square Footage' and 'Square Meters'. These two features contain the exact same information.

This is called **Multicollinearity**. When columns are highly correlated, the matrix $(\mathbf{X}^T\mathbf{X})$ becomes *ill-conditioned* (nearly singular). Its determinant gets extremely close to zero, meaning the computer struggles to calculate its inverse, causing the weights ($\beta$) to explode into massive, unstable numbers.

We can diagnose this using the **Condition Number**. A condition number near 1 means the matrix is perfectly stable (orthogonal). A condition number in the thousands or millions means the matrix is dangerously unstable.

In [ ]:
# Let's create an Unstable Design Matrix with Highly Correlated Features
feature_1 = np.random.rand(100)
# Feature 2 is almost identical to Feature 1, plus a tiny bit of noise
feature_2 = feature_1 + np.random.normal(0, 0.001, 100) 

X_unstable = np.column_stack((np.ones(100), feature_1, feature_2))
y_dummy = 3 + 4 * feature_1 + np.random.normal(0, 1, 100)

# Check the Condition Number of X^T X
XtX_unstable = X_unstable.T @ X_unstable
condition_number = np.linalg.cond(XtX_unstable)

print(f"Condition Number of X^T X: {condition_number:,.2f}")
print("A condition number this high indicates extreme multicollinearity!")

# Let's see what happens to the weights (Betas) when we force a standard inverse
try:
    unstable_betas = np.linalg.inv(XtX_unstable) @ X_unstable.T @ y_dummy
    print("\nCalculated Betas for [Intercept, Feature_1, Feature_2]:")
    print(np.round(unstable_betas, 2))
    print("Notice how the coefficients for F1 and F2 are massively inflated and opposing each other!")
except np.linalg.LinAlgError:
    print("The matrix is completely singular and cannot be inverted.")

## 12.4 Matrix Regularization (Ridge Regression)

If we have multicollinearity and our matrix is unstable, how do we rescue it? 
We use **Regularization** (also known in linear algebra as Tikhonov Regularization, and in statistics as Ridge Regression).

The instability comes from the fact that $(\mathbf{X}^T\mathbf{X})$ has values too close to zero on its diagonal. We can "prop up" the matrix by artificially adding a small positive number ($\lambda$) to every element on the main diagonal using the Identity matrix $\mathbf{I}$.

The regularized Least Squares equation becomes:
$$ \beta = (\mathbf{X}^T\mathbf{X} + \lambda\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y} $$

This small addition breaks the collinearity, drastically lowers the condition number, and stabilizes the weights, allowing the model to generalize much better to new data.

In [ ]:
# 1. Define the Lambda parameter (Regularization strength)
lambd = 0.5

# 2. Create the Identity matrix (same shape as X^T X)
I = np.eye(XtX_unstable.shape[0])

# We usually don't regularize the intercept term, so we set the top-left to 0
I[0, 0] = 0 

# 3. Add the penalty to the diagonal of X^T X
XtX_regularized = XtX_unstable + (lambd * I)

# 4. Re-calculate the Condition Number
new_condition_number = np.linalg.cond(XtX_regularized)

print(f"Old Condition Number : {condition_number:,.2f}")
print(f"New Condition Number : {new_condition_number:,.2f}")
print("The matrix is now incredibly stable!\n")

# 5. Solve for the new Regularized Betas
stable_betas = np.linalg.inv(XtX_regularized) @ X_unstable.T @ y_dummy

print("Old Unstable Betas:", np.round(unstable_betas, 2))
print("New Stable Betas  :", np.round(stable_betas, 2))
print("The regularized weights are now reasonable and evenly distributed between the correlated features.")

## 12.5 Chapter Summary

In Chapter 12, we extended the capabilities of the Least Squares algorithm:
- **Polynomial Fitting:** By explicitly transforming our input features (squaring, cubing) before building the design matrix, we can fit curved datasets using the exact same linear algebraic mechanics.
- **Multicollinearity:** When features are highly correlated, the matrix $(\mathbf{X}^T\mathbf{X})$ becomes ill-conditioned, leading to extreme sensitivity and exploding model weights. This is diagnosed via the Condition Number.
- **Regularization:** By adding a scaled Identity matrix $\lambda\mathbf{I}$ to $(\mathbf{X}^T\mathbf{X})$, we forcibly stabilize the diagonal. This trades a tiny bit of training accuracy (adding bias) for a massive increase in model stability and real-world robustness (reducing variance).